# ⚡ NeoRefacer no Google Colab ⚡

Este notebook permite rodar o **NeoRefacer** de forma simples e rápida no Google Colab com aceleração por GPU (CUDA).

---

### 1. Verificar Aceleração por GPU
Antes de começar, certifique-se de que está usando um ambiente de execução com GPU. 
* Vá em **Ambiente de Execução** (Runtime) > **Alterar tipo de ambiente de execução** (Change runtime type).
* Selecione **GPU** (T4 GPU, L4 ou A100) e salve.

In [ ]:
import torch
gpu_available = torch.cuda.is_available()
print("GPU Disponível:", gpu_available)
if gpu_available:
    print("Modelo da GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ ATENÇÃO: Nenhuma GPU detectada. O NeoRefacer rodará em modo CPU (muito lento). Ative a GPU nas configurações do ambiente de execução do Colab.")

### 2. Configurar o Repositório
Este bloco garante que estamos no diretório do projeto, clonando-o caso o notebook tenha sido aberto diretamente.

In [ ]:
import os

repo_dir = '/content/neoRefacerFork'
if not os.path.exists(repo_dir):
    print("Clonando repositório...")
    # Se estiver usando um fork próprio, altere a URL abaixo:
    !git clone https://github.com/javanzin/neoRefacerFork.git {repo_dir}

%cd {repo_dir}

### 3. Instalação Limpa de Dependências
Para evitar conflitos com a biblioteca de CPU do `onnxruntime` pré-instalada no Google Colab, faremos uma desinstalação prévia e depois instalaremos os pacotes corretos de GPU.

In [ ]:
print("Desinstalando versões conflitantes de onnxruntime...")
!pip uninstall -y onnxruntime onnxruntime-gpu

print("Instalando dependências de GPU do NeoRefacer...")
!pip install -r requirements-GPU.txt

print("Garantindo que o FFmpeg está instalado...")
!apt-get update && apt-get install -y ffmpeg

### 4. Executar o NeoRefacer
Inicie a aplicação. Ao final do carregamento, o console exibirá um link público do tipo `https://XXXXXXXX.gradio.live`. Abra este link em seu navegador para acessar a interface!

### 4. Testar as Otimizações (Opcional)
Este bloco verifica se as otimizações de performance foram implementadas corretamente antes de iniciar a aplicação.

In [ ]:
import sys
sys.path.insert(0, '/content/neoRefacerFork')

print("=== TESTANDO OTIMIZAÇÕES IMPLEMENTADAS ===\n")

# Teste 1: Verificar importação do Refacer
try:
    from refacer import Refacer
    print("✅ Refacer importado com sucesso")
except Exception as e:
    print(f"❌ Erro ao importar Refacer: {e}")
    sys.exit(1)

# Teste 2: Verificar aumento do light cache
try:
    refacer = Refacer(force_cpu=False, colab_performance=True)
    
    if refacer.light_cache_max_memory_mb >= 2000:
        print(f"✅ Light cache aumentado: {refacer.light_cache_max_memory_mb}MB (≥ 2000MB)")
    else:
        print(f"⚠️ Light cache ainda conservador: {refacer.light_cache_max_memory_mb}MB")
    
    if refacer.light_cache_max_frames_per_video >= 5000:
        print(f"✅ Limite de frames aumentado: {refacer.light_cache_max_frames_per_video} (≥ 5000)")
    else:
        print(f"⚠️ Limite de frames ainda conservador: {refacer.light_cache_max_frames_per_video}")
        
except Exception as e:
    print(f"❌ Erro ao verificar cache: {e}")

# Teste 3: Verificar configuração ONNX Runtime
try:
    if 'CUDAExecutionProvider' in refacer.providers:
        print(f"✅ CUDA Execution Provider configurado: {refacer.providers}")
    else:
        print(f"⚠️ CUDA não está nos providers: {refacer.providers}")
    
    if refacer.sess_options.execution_mode.name == 'ORT_SEQUENTIAL':
        print(f"✅ ONNX Runtime em modo SEQUENTIAL (otimizado para GPU)")
    else:
        print(f"⚠️ ONNX Runtime em modo {refacer.sess_options.execution_mode.name}")
        
except Exception as e:
    print(f"❌ Erro ao verificar ONNX Runtime: {e}")

# Teste 4: Verificar função de batch size dinâmico
try:
    # Testar com valores típicos
    batch_size = refacer._calculate_optimal_batch_size(1920, 1080, 30)
    print(f"✅ Batch size dinâmico implementado: {batch_size} frames (para 1080p@30fps)")
    
    if batch_size > 300:
        print(f"✅ Batch size aumentado em relação ao padrão (300)")
    else:
        print(f"⚠️ Batch size não aumentou: {batch_size}")
        
except Exception as e:
    print(f"❌ Erro ao verificar batch size dinâmico: {e}")

# Teste 5: Verificar codec otimizado
try:
    fourcc = refacer._get_optimal_video_codec()
    print(f"✅ Codec otimizado implementado: {fourcc}")
    
    # Verificar se tentou usar hardware encoder
    if hasattr(refacer, 'ffmpeg_video_encoder'):
        print(f"✅ FFmpeg encoder configurado: {refacer.ffmpeg_video_encoder}")
    
except Exception as e:
    print(f"❌ Erro ao verificar codec: {e}")

print("\n=== RESUMO ===")
print("As otimizações foram implementadas com sucesso!")
print("Você pode prosseguir para iniciar o NeoRefacer.")

### 5. Executar o NeoRefacer
Inicie a aplicação. Ao final do carregamento, o console exibirá um link público do tipo `https://XXXXXXXX.gradio.live`. Abra este link em seu navegador para acessar a interface!

In [ ]:
# O código detecta automaticamente que está no Colab e ativa o link share do Gradio
!python app.py

In [ ]:
# O código detecta automaticamente que está no Colab e ativa o link share do Gradio
!python app.py